[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Convolutional Neural Networks

A CNN is [DSP](../../Intro_DSP/README.md) that trains itself: convolution — the operation you mastered in [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — with the filter taps turned into **learnable weights**. We classify signals by their spectrograms and then *open the hood* to see what filters the network chose to learn.

## 0. Introduction

Why not use the MLP from the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb) on images/spectrograms? Because a dense layer on a 128×128 input needs millions of weights and must re-learn the same edge detector at every location. Convolution fixes both with two priors:

- **Locality** — nearby pixels/samples are related; a small kernel suffices.
- **Weight sharing** — a feature is the same feature *wherever* it appears, so one kernel slides everywhere (this is what makes the layer a convolution!).

## 1. Pre-requisites

- [Intro to ANN](../Intro_ANN/Intro_ANN.ipynb) — layers, backprop, the training loop.
- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — `nn.Module`, Dataloaders.
- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) Sessions 4–6 — convolution & the STFT/spectrogram.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy import signal as sig

torch.manual_seed(0)
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Convolution as a Learned Filter Bank* (~35 min)
**Goal:** map conv/pool/stride/receptive-field onto DSP concepts you already own.
**Builds on:** [ANN](../Intro_ANN/Intro_ANN.ipynb); [DSP Foundations](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb). &nbsp; **Feeds into:** Session 2 (train & inspect).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Convolution as a Learned Filter Bank</b></summary>

**Timing (~35 min).** 10 min why an MLP fails on images · 12 min the DSP translation table · 8 min the hand-made kernel demo · 5 min receptive fields.

**Open with the failure, not the solution.** Ask what happens if you feed a $128\times128$ image to the dense layer from the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb). The input is 16,384 numbers; a modest 512-unit first layer needs **8.4 million weights**. Then ask the deeper question: having learned an edge detector for the top-left corner, does that layer know anything about edges in the bottom-right? It does not — every location gets its own independent weights, and each must learn the same thing from scratch. Both problems have one fix.

**Name the two priors explicitly, because they are the entire architectural idea.** **Locality**: pixels far apart are weakly related, so a $3\times3$ kernel suffices. **Weight sharing**: a feature is the same feature wherever it appears, so one kernel slides everywhere. Then make the punchline explicit — *a layer with those two constraints imposed on a dense layer **is** a convolution*. Convolution is not an operation someone chose to import from signal processing; it is what a dense layer becomes once you demand translation equivariance.

**Spend real time on the translation table, since this audience already owns the right side of it.** Kernel = FIR taps. Feature map = filtered output. Channels = a filter bank. Stride = decimation. Pooling = nonlinear downsampling. Every term in the CNN vocabulary is a DSP term with a new name, and students who have done [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) find the whole architecture familiar rather than novel once they see this.

**The one honest wrinkle worth flagging: `nn.Conv2d` computes correlation, not convolution.** There is no kernel flip. For a *learned* kernel this is irrelevant — the network simply learns the flipped taps — but a student who hand-codes a kernel from a DSP textbook and expects matching output will be confused, and the confusion is not their fault. Say it once and move on.

**Use the $[1, 0, -1]$ demo to establish that a kernel is readable.** It is a first-difference along $x$, a discrete $\partial/\partial x$, a 2-D FIR high-pass in one direction — all the same statement. The point being set up for Session 2 is that **you can look at a kernel and say what it detects**, which is why the learned filters later in this notebook can be interpreted at all. Contrast with the MLP of the previous workshop, where a first-layer weight vector is an unreadable 1024-dimensional smear.

**Cover receptive field growth if time allows, because it explains depth.** Two stacked $3\times3$ layers see a $5\times5$ patch; three see $7\times7$; add a pooling layer and the field doubles. So depth is how a network gets from taps to edges to textures to shapes without ever using a large kernel. **Small kernels stacked deep beat one large kernel** — fewer parameters, more nonlinearity, same reach — which is the design lesson that produced VGG and everything after it.

**Close by framing Session 2 as an experiment rather than a demo.** We will train this on spectrograms and then *open the hood*. The claim to put on the board now, so it can be tested later: on spectrogram input, oriented kernels are frequency-trajectory detectors — a tilted edge fires on chirps, a horizontal one on steady tones. If that claim is right, the learned filter bank should look like something a signals engineer would have designed by hand.
</details>

## 2. Theory: the CNN Vocabulary, Translated from DSP

| CNN term | DSP translation |
|---|---|
| kernel / filter | FIR filter taps (2-D), **learned** |
| feature map | the filtered output signal |
| channels | a *filter bank* — many filters in parallel |
| stride | decimation / downsampling |
| pooling | nonlinear downsampling (max = "was the feature present anywhere here?") |
| receptive field | the region of input a deep unit can "see" — grows with depth |

💡 **Intuition.** A convolutional *layer* is a filter bank followed by a nonlinearity. Stacking layers composes filters into detectors of ever larger, ever more abstract patterns: taps → edges → textures → shapes. The network is a *hierarchical* filter bank whose every tap was chosen by gradient descent instead of by [Filter Design](../../Intro_DSP/Filter_Design.ipynb).

### 2.1. Seeing One Convolution

Before trusting `nn.Conv2d`, apply a hand-made kernel: a vertical-edge detector is just a 2-D FIR high-pass in one direction.

In [2]:
# A test image: a bright square on darkness
img = np.zeros((64, 64), dtype=np.float32)
img[16:48, 20:44] = 1.0

k_edge = np.array([[1, 0, -1]], dtype=np.float32)          # d/dx-ish kernel
edges = sig.convolve2d(img, k_edge, mode="same")

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("input")
axes[1].imshow(edges, cmap="RdBu"); axes[1].set_title("after [1, 0, −1]: vertical edges")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/2189497216.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** A solid white square goes in; two thin vertical stripes come out — red on the left edge, blue on the right — and the **interior is exactly zero**. Three numbers, $[1, 0, -1]$, deleted everything that was not a vertical transition.

**Read the kernel as a derivative and the output explains itself.** $[1,0,-1]$ computes $x[n-1] - x[n+1]$, a scaled first difference along the horizontal axis — a discrete $\partial/\partial x$. Inside the square all neighbours are equal, so the difference is zero. At the left edge the value jumps up, at the right edge it jumps down, and the two signs are why the stripes are opposite colours. The horizontal edges of the square are invisible because the kernel differences only in $x$; a transposed kernel would find those instead.

**The same three numbers are a high-pass FIR filter, which is the whole point of the session.** Its frequency response is $H(\omega) = 2j\sin\omega$: zero at DC, rising to a peak at Nyquist. Flat regions are DC and get annihilated; edges are broadband and survive. **"Edge detector" and "high-pass filter" are the same object described by two communities**, and everything in [Filter Design](../../Intro_DSP/Filter_Design.ipynb) applies unchanged.

**Note what is readable here, because it is what makes Session 2 possible.** You can *look* at $[1,0,-1]$ and say what it does. Compare a first-layer weight vector from the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb) — 1024 numbers with no spatial meaning, essentially uninterpretable. Kernels keep their geometry, so when we plot the *learned* kernels later, "this one is a tilted edge detector" is a statement you can actually make about a trained network.

**One implementation caveat worth knowing before students hand-code kernels.** `scipy.convolve2d` flips the kernel; `nn.Conv2d` **does not** — PyTorch computes cross-correlation and calls it convolution. For a learned kernel this makes no difference, since the network just learns the flipped taps. For a hand-specified one it flips the sign of the output, and it is a genuinely confusing first encounter if nobody says so.

**Finally, the setup for what follows.** `nn.Conv2d(1, 16, 3)` learns **sixteen** such $3\times3$ kernels simultaneously, plus biases — 160 parameters total. Nobody chooses the taps; backprop nudges each of the 144 weights exactly as in the ANN workshop, with the single difference that each weight is shared across every position in the image. That sharing is what makes it a convolution rather than a dense layer.

`nn.Conv2d(1, 16, 3)` learns **16 such 3×3 kernels at once** — plus their biases — and backprop nudges every tap, exactly as in the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb), just with shared weights.

---
### 🕐 Session 2 of 2 — *Train a CNN on Spectrograms* (~40 min)
**Goal:** classify chirps vs tones vs noise bursts from their spectrograms; visualize learned kernels.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Train a CNN on Spectrograms</b></summary>

**Timing (~40 min).** 8 min the task and the data · 7 min the architecture and parameter count · 10 min training · 15 min opening the hood.

**Frame the task as a signals-lab problem, because it is one.** Given a spectrogram, is the emitter a rising chirp, a pure tone, or a noise burst? The same pipeline classifies radar pulses, bird calls, and modulation schemes. Students who have done [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) Sessions 4–6 already know what each class looks like in time–frequency, which is exactly what makes this a good vehicle for interpretability later.

**Have the room read the three spectrograms before any model appears.** A chirp is a diagonal ridge; a tone is a horizontal line; a noise burst is a vertical block. Ask what a *hand-designed* detector would look like for each — a tilted matched filter, a horizontal one, a vertical one. Write those three answers on the board and leave them there. In twenty minutes we will plot the learned kernels and compare.

**The parameter count deserves a pause.** 136k parameters total, and the printout points out that a dense first layer on the same input would already need ~525k. But the more instructive comparison is inside the model: the two convolutional layers hold **4,800** parameters and do all the feature extraction, while the single dense layer holds **131,136** — 96% of the model. Ask where the parameters went. **Convolution is cheap; flattening into a dense layer is what costs.** That observation is why modern architectures replace the flatten with global average pooling, and it is worth naming.

**Do not skip the train/test split discussion just because PyTorch makes it one line.** The previous workshop reported training accuracy and this one reports test accuracy, which is a deliberate upgrade. Point at `idx[:700], idx[700:]` and say plainly: the 200 test spectrograms were never seen by the optimiser. That is the only number that means anything, and the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb) deliberately lacked it.

**Then be honest about the result, because 100% by epoch 1 is a warning, not a triumph.** The three classes are near-orthogonal in time–frequency, the SNR is generous, and the classes are perfectly balanced. **The task is too easy to distinguish good architectures from bad ones** — an MLP, a random forest on a few hand-crafted features, or a nearest-neighbour classifier on raw spectrograms would all do well. Say this out loud; a room that leaves believing "the CNN succeeded where simpler methods fail" has learned something false. The demo's job is interpretability, not benchmarking.

**Suggest the fix if anyone wants to push it.** Lower the SNR until accuracy degrades, add a fourth class that is genuinely confusable (a slow chirp versus a tone), or shrink the training set to 50 examples. The workshop then acquires a real experimental question, and the difference between architectures becomes measurable.

**Budget the most time for the kernel plot, because it is the session's actual payoff.** Compare the learned $3\times3$ filters against the three predictions on the board. Some are oriented — tilted, horizontal, vertical — which is exactly what the task demands. Others will look like noise, and it is important to say why rather than glossing over it: with a task this easy, the loss reaches zero before most filters are forced to specialise, so many remain near their initialisation. **Interpretability is partial, and honesty about which filters are readable matters more than a tidy story.**

**Close on the through-line the workshop opened with.** The network was not told about chirps, tones, matched filters, or orientation. It was given locality, weight sharing, and a gradient — and it recovered a filter bank a signals engineer would recognise. Then point forward: [Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) drop even the locality prior and let attention decide connectivity from the data.
</details>

## 3. Application: What Kind of Signal Is This?

A task straight from a signals lab: given a spectrogram, is the emitter a **rising chirp**, a **pure tone**, or a **noise burst**? (The same pipeline classifies radar pulses, bird calls, and modulation schemes.)

In [3]:
fs, T = 1024, 1.0
t = np.arange(0, T, 1 / fs)

def make_example(cls):
    if cls == 0:    # chirp
        f0 = rng.uniform(50, 150)
        x = sig.chirp(t, f0=f0, f1=f0 + rng.uniform(100, 250), t1=T)
    elif cls == 1:  # tone
        x = np.sin(2 * np.pi * rng.uniform(80, 350) * t)
    else:           # noise burst
        x = np.zeros_like(t)
        s = rng.integers(0, len(t) // 2)
        x[s:s + len(t) // 3] = rng.standard_normal(len(t) // 3)
    x = x + 0.3 * rng.standard_normal(len(t))
    _, _, S = sig.stft(x, fs=fs, nperseg=64)
    S = np.log1p(np.abs(S))[:32, :32]                     # 32×32 log-spectrogram
    return (S - S.mean()) / (S.std() + 1e-6)

X = np.stack([make_example(c % 3) for c in range(900)]).astype(np.float32)
y = np.array([c % 3 for c in range(900)])

fig, axes = plt.subplots(1, 3, figsize=(8, 2.6))
for ax, c, name in zip(axes, [0, 1, 2], ["chirp", "tone", "noise burst"]):
    ax.imshow(X[c], origin="lower", aspect="auto")
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/940485882.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three $32\times32$ log-spectrograms, and each class has a signature you can name without any model at all:

| class | time–frequency signature | the detector you would hand-design |
|---|---|---|
| chirp | a **diagonal** ridge climbing in frequency | a tilted matched filter |
| tone | a **horizontal** line at fixed frequency | a horizontal edge detector |
| noise burst | a **vertical** block, broadband but brief | a vertical edge detector |

**Write those three answers down now, because they are the prediction the workshop will test.** Session 1 claimed that on spectrogram input, oriented kernels are frequency-trajectory detectors. If that is right, the filters this network *learns* should look like the ones in the right-hand column — and we will plot them in a few cells and check.

**Note that the STFT did the hard work before the CNN sees anything.** In the raw time domain a chirp and a tone are both oscillating waveforms, distinguishable only by subtle phase evolution — hard for a small convolutional network. In time–frequency they become **geometric shapes**, and shape recognition is precisely what a CNN is built for. Choosing the representation is the modelling decision here; the architecture is downstream of it. That is a general lesson worth stating: a good transform can turn a hard learning problem into an easy one, and no amount of architecture search substitutes for it.

**Two preprocessing details are doing real work and are easy to skip past.** `log1p` compresses the enormous dynamic range of $|S|$ so that quiet structure is visible rather than being swamped by the loudest bin — the same reason spectrograms are conventionally shown in dB. And each example is standardized individually, so the network cannot cheat by reading absolute loudness; it must use shape. Remove either and the training curve changes noticeably.

**Now the honest framing, before the results arrive.** These three classes are close to orthogonal in this representation — diagonal, horizontal, vertical — with generous SNR (signal plus $0.3\sigma$ noise) and perfectly balanced classes by construction (`c % 3`). **This is a demonstration dataset, not a benchmark.** Expect it to be easy, and treat the accuracy it produces as evidence that the pipeline works rather than as evidence that CNNs beat alternatives here. A nearest-neighbour classifier on these raw spectrograms would also do well.

**The interesting experiments start where this one stops.** Drop the SNR until the classes overlap; add a fourth class that is genuinely confusable, such as a very slow chirp against a steady tone; or cut the training set to 50 examples and watch the parameter count start to matter. Any of those turns the notebook from a demonstration into an experiment with an outcome you cannot predict in advance.

In [4]:
class SpecCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32→16
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16→8
        )
        self.classify = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x):
        return self.classify(self.features(x))

model = SpecCNN()
print(sum(p.numel() for p in model.parameters()), "parameters",
      "(an MLP flattening 32×32 to 1024→512 would already need ~525k in its first layer)")

136131 parameters (an MLP flattening 32×32 to 1024→512 would already need ~525k in its first layer)


**What just happened.** **136,131 parameters** — against roughly 525,000 that a dense first layer alone would need on the same $32\times32$ input. But the comparison the printout makes is less interesting than the one *inside* the model, so break the total down:

| block | parameters | share |
|---|---|---|
| `Conv2d(1, 16, 3)` | $16 \times 9 + 16 = 160$ | 0.1% |
| `Conv2d(16, 32, 3)` | $32 \times 16 \times 9 + 32 = 4{,}640$ | 3.4% |
| `Linear(2048, 64)` | $2048 \times 64 + 64 = 131{,}136$ | **96.3%** |
| `Linear(64, 3)` | $195$ | 0.1% |

**All the feature extraction happens in 4,800 parameters; the flatten-and-classify head holds 96% of the model.** That is the number to sit with. Convolution is astonishingly cheap because a kernel is reused at every position — 160 weights process 1024 pixels — while a dense layer pays for every (input, output) pair separately.

**Which explains a real architectural trend rather than just being a fun fact.** Modern networks replace `Flatten → Linear` with **global average pooling**: average each of the 32 feature maps down to a single number, then a $32 \times 3$ linear layer. That would cut this model from 136k parameters to about 5k — a 27× reduction — with the additional benefit that the network then accepts any input size, since the pooling collapses whatever spatial extent it is given. If you want one concrete experiment from this cell, that is the one.

**Note where the two priors show up in the arithmetic.** *Locality*: each output pixel depends on a $3\times3$ patch, so the kernel is 9 numbers rather than 1024. *Weight sharing*: those 9 numbers are the same at every position, so the cost does not scale with image size at all. Together they take a layer from $O(N^2)$ parameters to $O(1)$. **The parameter count is where the priors become visible**, and it is worth pointing at the two conv lines and saying so.

**One more thing the shapes are telling you.** The comments track $32 \to 16 \to 8$: each `MaxPool2d(2)` halves the spatial dimensions while the channel count doubles, $1 \to 16 \to 32$. That trade — **spatial resolution down, feature richness up** — is the standard CNN shape, and the reason is receptive field. After two pool layers, one unit in the second conv layer sees a $10\times10$ patch of the original spectrogram; a chirp's diagonal ridge spans exactly that kind of extent, so the architecture is matched to the structure it needs to find.

In [5]:
Xt = torch.from_numpy(X).unsqueeze(1)          # (N, 1, 32, 32)
yt = torch.from_numpy(y)
idx = torch.randperm(len(Xt))
tr, te = idx[:700], idx[700:]

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(Xt[tr], yt[tr]), batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(6):
    model.train(); total = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
        total += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        acc = (model(Xt[te]).argmax(1) == yt[te]).float().mean()
    print(f"epoch {epoch}: loss {total / len(tr):.3f}   test acc {acc:.1%}")

epoch 0: loss 0.660   test acc 95.0%
epoch 1: loss 0.089   test acc 100.0%
epoch 2: loss 0.007   test acc 100.0%
epoch 3: loss 0.001   test acc 100.0%
epoch 4: loss 0.000   test acc 100.0%
epoch 5: loss 0.000   test acc 100.0%


**What just happened.** **95% test accuracy after one epoch, 100% after two**, and the loss falls to 0.000 by epoch 4. Note what is being reported: `acc` is computed on `Xt[te]` — 200 spectrograms held out of the optimiser entirely — so this is genuine test accuracy, an upgrade on the training accuracy the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb) reported.

**But 100% by epoch 1 is a warning, not a triumph, and it should be read as one.** The three classes are near-orthogonal in this representation — diagonal, horizontal, vertical — the SNR is generous, and the classes are perfectly balanced by construction. **The task is too easy to discriminate between architectures.** An MLP would do well here. So would a random forest on three hand-crafted features, or nearest-neighbour on the raw spectrograms. A room that leaves believing "the CNN succeeded where simpler methods fail" has learned something the experiment does not support.

**Say precisely what the result does establish, because it does establish something.** The pipeline is correct: the data generation, the tensor shapes, the loss, the optimiser, and the train/test split all work, and the network has learned features that transfer to unseen examples rather than memorising. That is exactly what you want to confirm before spending time on interpretability — which is this notebook's actual subject.

**The loss reaching 0.000 has a consequence that matters for the next few cells.** Once every example is classified with near-certainty, the gradient is essentially zero and **learning stops**. Filters that were already good enough get no further pressure to specialise, and filters that were never needed stay close to their random initialisation. When the kernel plot below shows a mix of clearly-oriented filters and noisy ones, this is why. An easy task does not force a network to use all of its capacity.

**Two things worth flagging in the training loop itself.** `model.train()` / `model.eval()` toggle dropout and batch-norm behaviour — neither is present here, so the calls are inert, but they are the correct habit and omitting them is a classic silent bug in models that do use those layers. And `torch.no_grad()` around evaluation is not optional at scale: without it, PyTorch builds a graph for every test batch and the memory is never freed.

**If you want the notebook to answer a question rather than confirm one, make the task harder.** Drop the SNR until the classes overlap, add a slow chirp that is genuinely confusable with a tone, or cut the training set to 50 examples. Any of those produces a learning curve with something to explain — and only then does the architecture comparison become a measurement rather than an assertion.

### 3.1. Open the Hood: What Did It Learn?

First-layer kernels are directly plottable — and on spectrogram input, oriented kernels are *frequency-trajectory detectors*: a tilted edge detector fires on chirps, a horizontal one on steady tones.

In [6]:
kernels = model.features[0].weight.detach().squeeze(1)   # (16, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(9, 2.6))
for ax, k in zip(axes.ravel(), kernels):
    ax.imshow(k, cmap="RdBu"); ax.axis("off")
plt.suptitle("The 16 learned 3×3 kernels — a filter bank nobody designed")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/4163168610.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Sixteen $3\times3$ kernels, 144 numbers in total, chosen entirely by gradient descent — and several of them are **readable**. Look for red and blue arranged along a direction: those are oriented edge detectors, and the direction of the red/blue split is the orientation the filter responds to.

**Compare them against the prediction made before training.** A chirp is a diagonal ridge, a tone is a horizontal line, a noise burst is a vertical block, so the filters a signals engineer would design are tilted, horizontal, and vertical matched filters. Several of the learned kernels are exactly that. **Nobody told the network about chirps, orientation, or matched filtering** — it was given locality, weight sharing, and a loss, and it recovered the filter bank from the task.

**Now the honest part, because a tidy story here would be a false one: not all sixteen are interpretable.** Several look like noise. That is not a defect in the plot, and it has a specific cause visible two cells above — the loss hit 0.000 by epoch 4. Once every example is classified with near-certainty the gradient vanishes, so **filters that were never needed simply stayed near their random initialisation**. An easy task does not force a network to use its capacity, and 16 filters is more than three near-orthogonal classes require. Expect roughly a handful of meaningful ones and a majority of passengers.

**Two further cautions on reading kernels, both of which trip up first attempts at interpretability.** The colour scale is per-panel, so a kernel with tiny weights is stretched to look as vivid as one with large weights — visual prominence is not importance. And a $3\times3$ filter can only see a $3\times3$ patch; the *interesting* features (a full chirp ridge) are assembled by layer 2 and the pooling, from combinations of these primitives. First-layer kernels are the alphabet, not the words.

**Still, the fact that this plot is meaningful at all is the architectural payoff.** Kernels retain their geometry, so "this one detects tilted edges" is a sentence you can say about a trained network. Try the same with the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb)'s first layer: each row is a 1024-dimensional vector with no spatial structure, and plotting it tells you nothing. **Weight sharing bought interpretability along with the parameter savings**, and that was not the reason it was introduced.

**One experiment that makes the point sharper.** Retrain on a harder version — lower SNR, or a fourth confusable class — and re-plot. With the loss unable to reach zero, the gradient keeps pushing, and a noticeably larger fraction of the sixteen filters specialise. The interpretability of a network is partly a function of how hard you made it work.

In [7]:
# And the feature maps for one chirp: which filters fire, and where?
with torch.no_grad():
    fmap = model.features[0](Xt[0:1]).squeeze(0)
fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
for ax, f in zip(axes.ravel(), fmap):
    ax.imshow(f, origin="lower", aspect="auto"); ax.axis("off")
plt.suptitle("Feature maps of a chirp after layer 1")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/200881441.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The same chirp, seen through all sixteen filters at once. Each panel is one feature map: the input convolved with one kernel. Filters whose orientation matches the chirp's diagonal ridge light up **along that ridge**; filters tuned to other orientations stay nearly flat. The bank has decomposed one spectrogram into sixteen simultaneous "is my feature here?" answers.

**This is the answer to what a feature map *is*, and it is worth saying in DSP terms.** It is the filtered signal — output of one FIR filter — and the collection of sixteen is the output of a **filter bank**, exactly as in [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb). The only difference from a designed bank is that these taps came from gradient descent rather than from a windowing method. Same object, different provenance.

**Note that the maps are *sparse*, and that the sparsity is ReLU doing its job.** Each panel is mostly dark with structure concentrated where the filter agrees with the input. Negative responses were clipped to zero, so a feature map answers "how strongly is my feature present here, and nowhere else". That selectivity is what makes the next layer's job tractable: it is combining a handful of localised detections, not re-processing a dense image.

**The maps also preserve *where*, which is the property that distinguishes convolution from a dense layer.** The chirp's ridge appears in the feature map at the same place it appears in the spectrogram. Shift the chirp later in time and every response shifts with it — **translation equivariance**, a direct consequence of weight sharing. The subsequent max-pooling then converts some of that equivariance into *invariance*, answering "was this feature present anywhere in this region?" rather than "exactly where was it?". Equivariance in the convolutions, invariance from the pooling — that pairing is the CNN's core design.

**Read the near-empty panels honestly, in light of the kernel plot above.** Some filters barely respond because they are genuinely tuned to a different orientation, which the network would need for the tone and noise-burst classes. Others barely respond because they never specialised at all — the loss hit zero before they were needed. **Both look identical in this plot**, and distinguishing them requires running the same visualisation on all three classes: a filter that is silent on every class is dead weight, while one that is silent here but active on tones is doing exactly its job. That is a five-minute experiment and a much stronger claim than either plot supports alone.

**The single sentence to leave with.** A CNN is a hierarchical filter bank whose every tap was chosen by gradient descent instead of by [Filter Design](../../Intro_DSP/Filter_Design.ipynb) — and, unusually for a neural network, you can look at it afterwards and largely say what it decided to build.

## 4. Conclusion

A CNN = hierarchical learned filter bank: locality and weight sharing shrink millions of weights to thousands, pooling buys translation tolerance, and the learned kernels are *readable* — on spectrograms they recover the matched filters a signal engineer would have designed.

---
## Where next

- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention removes even the locality prior and lets the data decide connectivity.
- [Filter Design](../../Intro_DSP/Filter_Design.ipynb) — the hand-designed baseline these kernels replace.
- [Scaling Neural Networks](../README.md#workshop-3--scaling-neural-networks-available) — this architecture, three orders of magnitude larger.